# D2.4 · Containment at machine speed

**Function D — Security Operations → The Incident Responder**  ·  *Security of AI*

---

**Risk.** Mass revocation takes down the business.

**Control.** Throttle → scope-reduce → reroute → force HITL → revoke → hard stop, in order.

**This lab.** Exercise the containment ladder in order, against a live misbehaving agent.

| | |
|---|---|
| Open-source tooling | agentgateway, Keycloak |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("D2.4"))

Containment at machine speed. A human in the containment path is a control that arrives after the damage, and the ratio is computable.

In [ ]:
from cybercommons import ir

for rate in (60, 300, 1200):
    r = ir.containment_race(agent_actions_per_min=rate, human_approval_minutes=8)
    print(f"{rate:>5} actions/min → {r['actions_during_manual_approval']:>8.0f} "
          f"during approval vs {r['actions_during_auto_containment']:>6.0f} automated "
          f"({r['ratio']}×)")
print("\n" + ir.containment_race(300, 8)["conclusion"])

Now the mechanism question, because 'contain it' is two very different actions.

In [ ]:
MECHANISMS = {
 "kill the process":     ("seconds", "it restarts; the credential still works"),
 "network quarantine":   ("seconds", "stops egress, not local damage"),
 "revoke the identity":  ("seconds", "the agent cannot act anywhere, even if it restarts"),
 "rotate the credential": ("minutes", "correct, but slower and breaks bystanders"),
}
for m, (speed, caveat) in MECHANISMS.items():
    print(f"{m:24s} {speed:8s} {caveat}")

### Expect

The three rates show 60–96× more actions during an eight-minute human approval than under automated containment, followed by the mechanism comparison.

### Your turn

Time your own revocation path end to end, from decision to the agent's next call failing. Most teams discover it is minutes, not seconds, and that the slow part is finding the right console.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/D2.4.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*